## Extract features from timeseries

In [ ]:
# Imports
import numpy as np
import pandas as pd
import os

In [ ]:
# Code from Nikolai Kühl (https://github.com/NK221088/Pupillometry_study/tree/main/Data_loading)
# Data structure
patient_data = pd.read_excel("../Pupilometri/Left_manually_cleaned_artefacts.xlsx", index_col=0, sheet_name=None)
patient_zero_indices = {sheet_name: np.where(patient_data[sheet_name].index == 0)[0][0] for sheet_name in patient_data.keys()} # Finding the first time index
patient_numeric_data = {sheet_name: patient_data[sheet_name].iloc[patient_zero_indices[sheet_name]:].apply(pd.to_numeric, errors='coerce') for sheet_name in patient_data.keys()}
patient_text_data = {sheet_name: patient_data[sheet_name].iloc[:patient_zero_indices[sheet_name]] for sheet_name in patient_data.keys()} # Extract text data -> Everything before first time index
all_patient_ids = list(patient_data.values())[0].keys() # Define all patient IDs


patient_raw_values = {sheet_name: patient_numeric_data[sheet_name] for sheet_name in patient_data.keys()}

patient_individual_raw_data = {patient_id: pd.concat([
    patient_raw_values[sheet_name][patient_id] if patient_id in patient_raw_values[sheet_name].columns else pd.Series(dtype='float64')
    for sheet_name in patient_data.keys()
    ], axis=1, keys=patient_data.keys()) for patient_id in all_patient_ids}

for key in patient_individual_raw_data:
    patient_individual_raw_data[key].index = (
        pd.to_numeric(patient_individual_raw_data[key].index, errors="coerce")
    )

    patient_individual_raw_data[key] = (
        patient_individual_raw_data[key]
        .loc[~patient_individual_raw_data[key].index.isna()]
        .sort_index()
    )


patient_individual_text_data = {patient_id: pd.concat([
    patient_text_data[sheet_name][patient_id] if patient_id in patient_text_data[sheet_name].columns else pd.Series(dtype='float64')
    for sheet_name in patient_text_data.keys()
    ], axis=1, keys=patient_text_data.keys()) for patient_id in all_patient_ids}

In [ ]:
# Code from Nikolai Kühl (https://github.com/NK221088/Pupillometry_study/tree/main/Data_loading)


##################################################################################
# Max PLR: (with slight modification used as PLR in Zarifkar et al)

def extract_max_PLR(data_dict, light_on_time, LOR_early_start_time):
    max_PLRs = {}

    for key, df in data_dict.items():
        interval = df.loc[
            (df.index >= light_on_time) &
            (df.index <= LOR_early_start_time)
        ]

        baseline = interval.iloc[0]
        min_value = interval.min()

        max_PLRs[key] = (baseline - min_value)/baseline

    return max_PLRs


##################################################################################
# Find 50 % numeric value

def extract_50_percent_timestamps(
    data_dict,
    patient_closest_timestamp_LOR_early_start_time,
    light_on_time,
    LOR_early_start_time,
    LOR_late_end_time
):
    _50_per_PLR_times = {}
    _50_per_LOR_times = {}

    for key, df in data_dict.items():
        interval = df.loc[:patient_closest_timestamp_LOR_early_start_time].dropna(axis=1)

        PLR_interval = df.loc[
            (df.index >= light_on_time) &
            (df.index <= LOR_early_start_time)
        ].dropna(axis=1)

        LOR_interval = df.loc[
            (df.index >= LOR_early_start_time) &
            (df.index <= LOR_late_end_time)
        ].dropna(axis=1)

        max_value = interval.max()
        _50_value = max_value * 0.5

        _50_per_PLR_times[key] = (
            (PLR_interval - _50_value).abs().idxmin()
        )

        _50_per_LOR_times[key] = (
            (LOR_interval - _50_value).abs().idxmin()
        )

    return _50_per_PLR_times, _50_per_LOR_times



##################################################################################
# LOR Early Gradient:

def extract_LOR_early_gradients(
    data_dict,
    LOR_early_start_time,
    LOR_early_end_time,
    ):
    LOR_early_gradients = {}

    for key, df in data_dict.items():
        interval = df.loc[
            (df.index >= LOR_early_start_time) &
            (df.index <= LOR_early_end_time)
        ]

        timespan = interval.index[-1] - interval.index[0]
        movement = (interval.max() - interval.iloc[0])

        LOR_early_gradients[key] = movement / timespan

    return LOR_early_gradients

##################################################################################
# LOR late Gradient:

def extract_LOR_late_gradients(
    data_dict,
    LOR_late_start_time,
    LOR_late_end_time,
    ):
    LOR_late_gradients = {}

    for key, df in data_dict.items():
        interval = df.loc[
            (df.index >= LOR_late_start_time) &
            (df.index <= LOR_late_end_time)
        ]

        timespan = interval.index[-1] - interval.index[0]
        movement = (interval.max() - interval.iloc[0])

        LOR_late_gradients[key] = movement / timespan

    return LOR_late_gradients



In [ ]:
# 50% time stamp resembling R  (Not totally identicale)
def extract_T50_all(
    data_dict,
    patient_closest_timestamp_LOR_early_start_time,
    light_on_time,
    LOR_early_start_time,
    LOR_early_end_time,
    LOR_late_start_time,
    LOR_late_end_time
):
    _50_per_PLR_times = {}
    _50_per_LOR_times = {}
    _50_per_LOR_early_times = {}
    _50_per_LOR_late_times = {}

    for key, df in data_dict.items():

        # BASELINE & GLOBAL MIN
  
        baseline_interval = df.loc[:patient_closest_timestamp_LOR_early_start_time].dropna(axis=1)
        baseline = baseline_interval.max()

        constriction_interval = df.loc[
            (df.index >= light_on_time) &
            (df.index <= LOR_early_end_time)
        ].dropna(axis=1)

        minimum = constriction_interval.min()

  
        # GLOBAL T50 
   
        t50_global_value = minimum + 0.5 * (baseline - minimum)

        LOR_interval = df.loc[
            (df.index >= LOR_early_start_time) &
            (df.index <= LOR_late_end_time)
        ].dropna(axis=1)

        _50_per_LOR_times[key] = (
            (LOR_interval - t50_global_value).abs().idxmin()
        )


        # PLR (CONSTRICTION, MIN/MAX-BASED)
       
        PLR_interval = df.loc[
            (df.index >= light_on_time) &
            (df.index <= LOR_early_start_time)
        ].dropna(axis=1)

        if not PLR_interval.empty:

            # get max (baseline) and min (peak constriction)
            max_PLR = PLR_interval.max()   # baseline (before constriction)
            min_PLR = PLR_interval.min()   # strongest constriction

            # compute T50 value
            # (halfway from max → min)
            t50_PLR_value = max_PLR - 0.5 * (max_PLR - min_PLR)

            # find closest time
            _50_per_PLR_times[key] = (
                (PLR_interval - t50_PLR_value).abs().idxmin()
            )

        else:
            _50_per_PLR_times[key] = None


        # EARLY LOR (LOCAL DILATION)
        LOR_early_interval = df.loc[
            (df.index >= LOR_early_start_time) &
            (df.index <= LOR_early_end_time)
        ].dropna(axis=1)

        start_early = LOR_early_interval.iloc[0]
        end_early = LOR_early_interval.iloc[-1]

        t50_early_value = start_early + 0.5 * (end_early - start_early)

        _50_per_LOR_early_times[key] = (
            (LOR_early_interval - t50_early_value).abs().idxmin()
        )

        # LATE LOR (MIN/MAX-BASED, DELAYED START)
        LOR_late_interval = df.loc[
            (df.index >= LOR_late_start_time) &
            (df.index <= LOR_late_end_time)
        ].dropna(axis=1)

        if not LOR_late_interval.empty:

            # find min + max
            min_late = LOR_late_interval.min()
            max_late = LOR_late_interval.max()

            #compute T50 value
            t50_late_value = min_late + 0.5 * (max_late - min_late)


            # find closest time to T50
            if not LOR_late_interval.empty:
                _50_per_LOR_late_times[key] = (
                    (LOR_late_interval - t50_late_value).abs().idxmin()
                )
            else:
                _50_per_LOR_late_times[key] = None

        else:
            _50_per_LOR_late_times[key] = None

    return (
        _50_per_PLR_times,
        _50_per_LOR_times,
        _50_per_LOR_early_times,
        _50_per_LOR_late_times
    )



def convert_nested_dict_to_series(nested_dict):

    converted = {}

    for patient_id, metric_dict in nested_dict.items():
        converted[patient_id] = pd.Series(metric_dict)

    return converted

# Pupil Dilation(LOR) from Zarifkar et al - Inspired from Nicolai
def extract_pupil_dilation(
    data_dict,
    LOR_early_start_time,
    LOR_late_end_time
):

    pupil_dilation = {}

    for key, df in data_dict.items():

        # Interval AFTER start of early LOR
        interval = df.loc[
            (df.index >= LOR_early_start_time) &
            (df.index <= LOR_late_end_time)
        ]

        # Ensure numeric
        interval = interval.select_dtypes(include=[np.number])

        if interval.empty:
            pupil_dilation[key] = np.nan
            continue

        dilation_per_column = {}

        for col in interval.columns:

            series = interval[col].dropna()

            if series.empty:
                dilation_per_column[col] = np.nan
                continue

            # Initial value at start of early LOR
            initial_value = series.iloc[0]

            # Max value AFTER that
            max_value = series.max()

            # Avoid division by zero
            if initial_value == 0:
                dilation_per_column[col] = np.nan
            else:
                dilation_per_column[col] = (
                    (max_value - initial_value) / initial_value
                )

        pupil_dilation[key] = dilation_per_column

    return pupil_dilation

#### Merge it with NPi and Date of examination

In [ ]:
# Code from Nikolai Kühl (https://github.com/NK221088/Pupillometry_study/tree/main/Data_loading) with slight changes

NPI_data = pd.read_excel("../Pupilometri/NPi_Light_off_redcapdata.xlsx")
merging_columns = [
    "date_examination",
    "light_off_performed",
    "npi_left",
    "npi_right",
]
for column in merging_columns:
    if column == "date_examination":
        NPI_data[f"{column}_merged"] = (
            NPI_data[f"{column}"]
            .combine_first(NPI_data[f"date_of_examination_2"])
        )
    else:
        NPI_data[f"{column}_merged"] = (
            NPI_data[f"{column}"]
            .combine_first(NPI_data[f"{column}_2"])
        )

columns_to_keep = ["record_id", "redcap_repeat_instance"] + [f"{col}_merged" for col in merging_columns]
NPI_data_cleaned = NPI_data[columns_to_keep] # Remove suffix from light_off_performed_merged column
NPI_data_cleaned["light_off_performed_merged"] = (
    NPI_data_cleaned["light_off_performed_merged"]
    .str.replace(r"\s*\(.*\)$", "", regex=True)
)
NPI_data_cleaned["redcap_repeat_instance"] = ( # Ensure first recording is marked as 0
    NPI_data_cleaned["redcap_repeat_instance"]
    .fillna(0)
)
mask = NPI_data_cleaned["record_id"] == 213
cols = ["npi_left_merged"]

NPI_data_cleaned.loc[mask, cols] = (
    NPI_data_cleaned.loc[mask, cols].replace(0, np.nan)
)
NPI_data_cleaned["redcap_repeat_instance"]  += 1 # Shift follow-up numbers to start from 1
NPI_data_cleaned = NPI_data_cleaned[
    NPI_data_cleaned["light_off_performed_merged"] == "Yes"
]
NPI_data_cleaned = (
    NPI_data_cleaned
    .sort_values(
        by=["record_id", "redcap_repeat_instance"],
        ascending=[True, True]
    )
)
NPI_data_cleaned["redcap_repeat_instance"] = (
    NPI_data_cleaned
    .groupby("record_id")
    .cumcount()
    .add(1)
)

In [ ]:
# Code from Nikolai Kühl (https://github.com/NK221088/Pupillometry_study/tree/main/Data_loading) with slight changes

zero_start_time = 0
light_on_time = 3
LOR_early_start_time = 6
LOR_early_end_time = 8
LOR_late_start_time = 8
LOR_late_end_time = 13

time_stamps = patient_individual_raw_data[1].index.astype(float)
patient_closest_timestamp_LOR_early_start_time = (
    patient_individual_raw_data[1]
    .index
    .to_series()
    .sub(LOR_early_start_time)
    .abs()
    .idxmin()
)


pupil_dilation =extract_pupil_dilation(
    patient_individual_raw_data,
    LOR_early_start_time,
    LOR_late_end_time
)
max_PLRs = extract_max_PLR(patient_individual_raw_data, light_on_time, LOR_early_start_time)
half_per_PLR_times, half_per_LOR_times,half_early,half_late = extract_T50_all(patient_individual_raw_data, patient_closest_timestamp_LOR_early_start_time, light_on_time,LOR_early_start_time,LOR_early_end_time,LOR_late_start_time,LOR_late_end_time
)
LOR_early_gradients = extract_LOR_early_gradients(patient_individual_raw_data, LOR_early_start_time, LOR_early_end_time)
LOR_late_gradients = extract_LOR_late_gradients(patient_individual_raw_data, LOR_late_start_time, LOR_late_end_time)


def dict_to_metric_df(metric_dict, value_name):
    return (
        pd.concat(metric_dict,
                  names=["record_id", "redcap_repeat_instance"])
          .rename(value_name)
          .reset_index()
    )

metric_dfs = [
    dict_to_metric_df(max_PLRs, "pupil_constriction"),
    dict_to_metric_df(convert_nested_dict_to_series(pupil_dilation),"pupil_dilation"),
    dict_to_metric_df(LOR_early_gradients, "LOR_early_gradient"),
    dict_to_metric_df(LOR_late_gradients, "LOR_late_gradient"),
]
metric_dfs += [
    dict_to_metric_df(half_per_PLR_times, "50pct_PLR_time"),
    dict_to_metric_df(half_per_LOR_times, "50pct_LOR_time"),
    dict_to_metric_df(half_early, "50pct_early"),
    dict_to_metric_df(half_late, "50pct_late")
]


seconds = {
    record_id: df.loc["SECONDS"]
    for record_id, df in patient_individual_text_data.items()
    if "SECONDS" in df.index
}
seconds_df = dict_to_metric_df(
    seconds,
    "SECONDS"
)

metric_dfs += [seconds_df]

four = {
    record_id: df.loc["FOUR"]
    for record_id, df in patient_individual_text_data.items()
    if "FOUR" in df.index
}
four_df = dict_to_metric_df(
    four,
    "FOUR"
)

metric_dfs += [four_df]

for i in range(len(metric_dfs)):
    metric_dfs[i]["redcap_repeat_instance"] = (
        metric_dfs[i]["redcap_repeat_instance"].astype(str)
    )
from functools import reduce

metrics_df = reduce(
    lambda l, r: l.merge(
        r,
        on=["record_id", "redcap_repeat_instance"],
        how="outer"
    ),
    metric_dfs
)

metrics_df["redcap_repeat_instance"] = (
    metrics_df["redcap_repeat_instance"]
    .str.extract(r'(\d+)$')
    .fillna(0)
    .astype(int)
    + 1
)




In [27]:
metrics_df = metrics_df.merge(
    NPI_data_cleaned[
        ["record_id", "redcap_repeat_instance", "date_examination_merged","npi_left_merged"]
    ],
    on=["record_id", "redcap_repeat_instance"],
    how="left"
)

metrics_df = metrics_df.sort_values(
    by=["record_id", "redcap_repeat_instance"]
)

In [28]:
df_npi_featues = pd.read_csv("../Modified_Pupilometri/NPImeaures.csv")
df_npi_featues = df_npi_featues[df_npi_featues["eye"] == "left"]


In [29]:
df_npi_featues['redcap_repeat_instance'] = (
    df_npi_featues['redcap_repeat_instance']
    .replace('', 0)
    .fillna(0)
)

df_npi_featues['redcap_repeat_instance'] += 1
df_npi_featues["record_id"] = df_npi_featues["record_id"].astype(int)

In [30]:
result = pd.merge(
    df_npi_featues,
    metrics_df,
    on=['record_id', 'redcap_repeat_instance'], 
    how='outer'
)
result = result.drop(columns=["Unnamed: 0"])

In [32]:
result.to_csv(
    os.path.join('../Modified_Pupilometri/', f'pupilometry_features_left.csv'))


### Merging left and right into a worst eye:

In [7]:
import pandas as pd
import numpy as np
import os

left = pd.read_csv("../Modified_Pupilometri/pupilometry_features_left_noCHerror.csv")
right = pd.read_csv("../Modified_Pupilometri/pupilometry_features_right_noCHerror.csv")

record_ids = pd.unique(pd.concat([left["record_id"], right["record_id"]]))

full_index = pd.MultiIndex.from_product(
    [record_ids, range(1, 21)],
    names=["record_id", "redcap_repeat_instance"]
)

panel = full_index.to_frame(index=False)

# merge NPi and CH
panel = panel.merge(
    left[["record_id", "redcap_repeat_instance", "npi", "ch", "pupil_size"]],
    on=["record_id", "redcap_repeat_instance"],
    how="left"
).rename(columns={
    "ch": "ch_left",
    "pupil_size": "pupil_size_left",
    "npi": "npi_left",
})

panel = panel.merge(
    right[["record_id", "redcap_repeat_instance", "npi", "ch", "pupil_size"]],
    on=["record_id", "redcap_repeat_instance"],
    how="left"
).rename(columns={
    "ch": "ch_right",
    "pupil_size": "pupil_size_right",
    "npi": "npi_right",
})

# Choose worst eye
def choose_eye(r):
    lnpi = r["npi_left"]
    rnpi = r["npi_right"]

    if pd.notna(lnpi) and pd.isna(rnpi):
        return "left"
    if pd.notna(rnpi) and pd.isna(lnpi):
        return "right"
    if pd.isna(lnpi) and pd.isna(rnpi):
        return np.nan

    if lnpi < rnpi:
        return "left"
    if rnpi < lnpi:
        return "right"

    lch = r["ch_left"]
    rch = r["ch_right"]

    if pd.notna(lch) and pd.notna(rch):
        if lch < rch:
            return "left"
        if rch < lch:
            return "right"

    lmin = r["pupil_size_left"]
    rmin = r["pupil_size_right"]

    if pd.notna(lmin) and pd.notna(rmin):
        return "left" if lmin < rmin else "right"

    return "right"

panel["worst_eye"] = panel.apply(choose_eye, axis=1)

features = [
    "arousal_gradient",
    "pupil_constriction",
    "LOR_early_gradient",
    "LOR_late_gradient",
    "50pct_PLR_time",
    "50pct_LOR_time",
    "min_size",
    "max_size",
    "ch",
    "const_velocity",
    "dilat_velocity",
    "max_const_velocity",
    "latency",
    "pupil_dilation",
    "SECONDS",
    "FOUR",
    "date_examination_merged"
]

rows = []

for _, r in panel.iterrows():
    rid = r["record_id"]
    inst = r["redcap_repeat_instance"]
    eye = r["worst_eye"]

    row = {
        "record_id": rid,
        "redcap_repeat_instance": inst,
        "worst_eye": eye,
        "worst_npi": np.nan
    }

    if eye == "left":
        match = left[
            (left["record_id"] == rid) &
            (left["redcap_repeat_instance"] == inst)
        ]
        if not match.empty:
            m = match.iloc[0]
            row["worst_npi"] = m["npi"]
            for f in features:
                row[f] = m.get(f)

    elif eye == "right":
        match = right[
            (right["record_id"] == rid) &
            (right["redcap_repeat_instance"] == inst)
        ]
        if not match.empty:
            m = match.iloc[0]
            row["worst_npi"] = m["npi"]
            for f in features:
                row[f] = m.get(f)

    for f in features:
        row.setdefault(f, np.nan)

    rows.append(row)

out = pd.DataFrame(rows)

final_cols = [
    "record_id",
    "redcap_repeat_instance",
    "worst_eye",
    "worst_npi"
] + features

out = out[final_cols]

out.to_csv(
    os.path.join("../Modified_Pupilometri", "pupilometry_features_worst_eye.csv"),
    index=False
)

print("DONE")

DONE
